# YOLO11s Optimization v1 (Notebook-first)

This notebook runs a full validation pipeline with:
- relative paths
- confidence calibration after training
- baseline vs FP16 inference comparison
- per-instance report, totals report, IoU visualizations


In [2]:
from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Dict, List, Set, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO

In [3]:
def discover_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    while p != p.parent:
        if (p / '.git').exists() and (p / 'v2').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not find repo root with .git and v2 folder')

REPO_ROOT = discover_repo_root()
print('Repo root:', REPO_ROOT)

CFG = {
    'model_weights': 'v2/runs/segment/placentas_v11_aug_v2/weights/best.pt',
    'data_yaml': 'v2/data_v2.yaml',
    'img_w': 640,
    'img_h': 640,
    'retina_masks': True,
    'iou_threshold': 0.5,
    # Corrected 2026-09-07: original images are 4140x3096 (4:3), stretched to 640x640 (1:1)
    # by the Roboflow export, which is anisotropic (X compressed 6.47x, Y compressed 4.84x).
    # The 50um/72px scale bar was measured horizontally (X axis) on the stretched 640x640
    # image, so (50/72)**2 implicitly assumed square 640-space pixels, which is wrong.
    # Correction factor = H_native/W_native = 3096/4140 = 0.747826 (validated independently
    # by direct pixel measurement on the native-resolution scale bar, agreement within 0.25%).
    'area_factor': (50 / 72) ** 2 * (3096 / 4140),
    'conf_candidates': [round(x, 2) for x in np.arange(0.20, 0.71, 0.02)],
    'output_root': 'v2_yolo11s_opt_v1/artifacts',
}

weights_path = (REPO_ROOT / CFG['model_weights']).resolve()
data_yaml_path = (REPO_ROOT / CFG['data_yaml']).resolve()
out_root = (REPO_ROOT / CFG['output_root']).resolve()
out_reports = out_root / 'reports'
out_iou = out_root / 'iou_viz'
out_bench = out_root / 'benchmarks'
for p in [out_root, out_reports, out_iou, out_bench]:
    p.mkdir(parents=True, exist_ok=True)

with data_yaml_path.open('r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

dataset_root = Path(data_cfg['path'])
if not dataset_root.is_absolute():
    dataset_root = (REPO_ROOT / dataset_root).resolve()

val_images = (dataset_root / data_cfg['val']).resolve()
val_labels = (dataset_root / 'valid' / 'labels').resolve()

print('weights:', weights_path)
print('val_images:', val_images)
print('val_labels:', val_labels)
print('out_root:', out_root)

Repo root: D:\projeto_placentas_clayton\dev\projeto-placentas
weights: D:\projeto_placentas_clayton\dev\projeto-placentas\v2\runs\segment\placentas_v11_aug_v2\weights\best.pt
val_images: D:\projeto_placentas_clayton\dataset_v2_ready_for_yolo\valid\images
val_labels: D:\projeto_placentas_clayton\dataset_v2_ready_for_yolo\valid\labels
out_root: D:\projeto_placentas_clayton\dev\projeto-placentas\v2_yolo11s_opt_v1\artifacts


In [4]:
def parse_gt_masks(label_path: Path, img_w: int, img_h: int) -> List[np.ndarray]:
    masks: List[np.ndarray] = []
    if not label_path.exists():
        return masks
    with label_path.open('r', encoding='utf-8') as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            if len(parts) <= 1:
                continue
            poly = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
            poly[:, 0] *= img_w
            poly[:, 1] *= img_h
            m = np.zeros((img_h, img_w), dtype=np.uint8)
            cv2.fillPoly(m, [poly.astype(np.int32)], 1)
            masks.append(m)
    return masks


def masks_from_result(r) -> List[np.ndarray]:
    if r.masks is None:
        return []
    return [(mask > 0.5).astype(np.uint8) for mask in r.masks.data.cpu().numpy()]


def iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    inter = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return float(inter) / float(union) if union > 0 else 0.0


def greedy_matches(ai_masks: List[np.ndarray], gt_masks: List[np.ndarray], iou_thr: float):
    candidates: List[Tuple[float, int, int]] = []
    for ai_idx, ai_mask in enumerate(ai_masks):
        for gt_idx, gt_mask in enumerate(gt_masks):
            score = iou(ai_mask, gt_mask)
            if score >= iou_thr:
                candidates.append((score, ai_idx, gt_idx))
    candidates.sort(key=lambda x: x[0], reverse=True)

    matches: List[Tuple[int, int, float]] = []
    used_ai: Set[int] = set()
    used_gt: Set[int] = set()
    for score, ai_idx, gt_idx in candidates:
        if ai_idx in used_ai or gt_idx in used_gt:
            continue
        used_ai.add(ai_idx)
        used_gt.add(gt_idx)
        matches.append((ai_idx, gt_idx, score))
    return matches, used_ai, used_gt

In [5]:
def evaluate_conf(model: YOLO, conf: float, half: bool = False) -> Dict[str, float]:
    results = model.predict(
        source=str(val_images),
        conf=conf,
        imgsz=CFG['img_w'],
        retina_masks=CFG['retina_masks'],
        half=half,
        verbose=False,
    )

    tp = fp = fn = 0
    ious: List[float] = []
    gt_total_area = ai_total_area = 0

    for r in results:
        img_name = Path(r.path).name
        label_path = val_labels / (Path(img_name).stem + '.txt')

        gt_masks = parse_gt_masks(label_path, CFG['img_w'], CFG['img_h'])
        ai_masks = masks_from_result(r)
        gt_total_area += sum(int(m.sum()) for m in gt_masks)
        ai_total_area += sum(int(m.sum()) for m in ai_masks)

        matches, used_ai, used_gt = greedy_matches(ai_masks, gt_masks, CFG['iou_threshold'])
        tp += len(matches)
        fp += len(ai_masks) - len(used_ai)
        fn += len(gt_masks) - len(used_gt)
        ious.extend([m[2] for m in matches])

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    mean_iou = float(np.mean(ious)) if ious else 0.0

    area_rel_error = abs(ai_total_area - gt_total_area) / max(gt_total_area, 1)
    # Prioritize count-quality (F1), then mask-shape quality, then area consistency.
    score = (0.6 * f1) + (0.3 * mean_iou) + (0.1 * (1.0 - area_rel_error))

    return {
        'conf': conf,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'mean_iou': mean_iou,
        'gt_total_area_px': gt_total_area,
        'ai_total_area_px': ai_total_area,
        'area_rel_error': area_rel_error,
        'score': score,
    }


model = YOLO(str(weights_path))
rows = [evaluate_conf(model, conf=c, half=False) for c in CFG['conf_candidates']]
conf_df = pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)
conf_df.head(10)

,conf,tp,fp,fn,precision,recall,f1,mean_iou,gt_total_area_px,ai_total_area_px,area_rel_error,score
0,0.46,743,77,109,0.906098,0.872066,0.888756,0.880925,4105301,4130657,0.006176,0.896913
1,0.48,735,70,117,0.913043,0.862676,0.887145,0.881844,4105301,3986951,0.028829,0.893958
2,0.50,731,59,121,0.925316,0.857981,0.890378,0.881317,4105301,3910632,0.047419,0.893880
3,0.44,752,92,100,0.890995,0.882629,0.886792,0.879184,4105301,4252115,0.035762,0.892254
4,0.42,759,102,93,0.881533,0.890845,0.886165,0.878782,4105301,4328091,0.054269,0.889906
5,0.52,720,49,132,0.936281,0.845070,0.888341,0.881021,4105301,3761978,0.083629,0.888948
6,0.54,711,42,141,0.944223,0.834507,0.885981,0.882909,4105301,3673849,0.105096,0.885952
7,0.40,764,117,88,0.867196,0.896714,0.881708,0.880303,4105301,4510038,0.098589,0.883257
8,0.38,768,124,84,0.860987,0.901408,0.880734,0.880072,4105301,4555546,0.109674,0.881495
9,0.56,694,33,158,0.954608,0.814554,0.879037,0.884287,4105301,3549101,0.135483,0.879160


In [6]:
conf_sweep_csv = out_bench / 'conf_sweep.csv'
conf_df.to_csv(conf_sweep_csv, index=False)

BEST_CONF = float(conf_df.iloc[0]['conf'])
print(f'Best confidence: {BEST_CONF:.4f}')
print(f'Confidence sweep saved: {conf_sweep_csv}')

summary_json = out_bench / 'selected_confidence.json'
summary_json.write_text(
    json.dumps({'best_conf': BEST_CONF, 'method': 'val_sweep'}, indent=2),
    encoding='utf-8',
)
summary_json

Best confidence: 0.4600
Confidence sweep saved: D:\projeto_placentas_clayton\dev\projeto-placentas\v2_yolo11s_opt_v1\artifacts\benchmarks\conf_sweep.csv


WindowsPath('D:/projeto_placentas_clayton/dev/projeto-placentas/v2_yolo11s_opt_v1/artifacts/benchmarks/selected_confidence.json')

In [7]:
def run_reports(tag: str, conf: float, half: bool) -> Tuple[Path, Path]:
    model_local = YOLO(str(weights_path))
    results = model_local.predict(
        source=str(val_images),
        conf=conf,
        imgsz=CFG['img_w'],
        retina_masks=CFG['retina_masks'],
        half=half,
        verbose=False,
    )

    instance_rows = []
    totals_rows = []
    for r in results:
        img_name = Path(r.path).name
        label_path = val_labels / (Path(img_name).stem + '.txt')

        gt_masks = parse_gt_masks(label_path, CFG['img_w'], CFG['img_h'])
        ai_masks = masks_from_result(r)
        matches, used_ai, used_gt = greedy_matches(ai_masks, gt_masks, CFG['iou_threshold'])

        gt_combined_area = int(sum(int(m.sum()) for m in gt_masks))
        ai_combined_area = int(sum(int(m.sum()) for m in ai_masks))

        totals_rows.append({
            'Image': img_name,
            'GT_Count': len(gt_masks),
            'AI_Count': len(ai_masks),
            'Matched_Count': len(matches),
            'FP_Count': len(ai_masks) - len(matches),
            'FN_Count': len(gt_masks) - len(matches),
            'GT_Area_px': gt_combined_area,
            'AI_Area_px': ai_combined_area,
            'GT_Area_um2': round(gt_combined_area * CFG['area_factor'], 4),
            'AI_Area_um2': round(ai_combined_area * CFG['area_factor'], 4),
            'Conf': conf,
            'Half_Mode': half,
            'Tag': tag,
        })

        for ai_idx, gt_idx, score in matches:
            gt_area = int(gt_masks[gt_idx].sum())
            ai_area = int(ai_masks[ai_idx].sum())
            instance_rows.append({
                'Image': img_name,
                'Match_Type': 'TP',
                'AI_Index': ai_idx,
                'GT_Index': gt_idx,
                'IoU': round(score, 6),
                'GT_Area_px': gt_area,
                'AI_Area_px': ai_area,
                'GT_Area_um2': round(gt_area * CFG['area_factor'], 4),
                'AI_Area_um2': round(ai_area * CFG['area_factor'], 4),
                'Conf': conf,
                'Half_Mode': half,
                'Tag': tag,
            })

        for ai_idx, ai_mask in enumerate(ai_masks):
            if ai_idx in used_ai:
                continue
            ai_area = int(ai_mask.sum())
            instance_rows.append({
                'Image': img_name,
                'Match_Type': 'FP',
                'AI_Index': ai_idx,
                'GT_Index': -1,
                'IoU': 0.0,
                'GT_Area_px': 0,
                'AI_Area_px': ai_area,
                'GT_Area_um2': 0.0,
                'AI_Area_um2': round(ai_area * CFG['area_factor'], 4),
                'Conf': conf,
                'Half_Mode': half,
                'Tag': tag,
            })

        for gt_idx, gt_mask in enumerate(gt_masks):
            if gt_idx in used_gt:
                continue
            gt_area = int(gt_mask.sum())
            instance_rows.append({
                'Image': img_name,
                'Match_Type': 'FN',
                'AI_Index': -1,
                'GT_Index': gt_idx,
                'IoU': 0.0,
                'GT_Area_px': gt_area,
                'AI_Area_px': 0,
                'GT_Area_um2': round(gt_area * CFG['area_factor'], 4),
                'AI_Area_um2': 0.0,
                'Conf': conf,
                'Half_Mode': half,
                'Tag': tag,
            })

    instance_path = out_reports / f'placenta_instance_report_{tag}.csv'
    totals_path = out_reports / f'placenta_totals_report_{tag}.csv'
    pd.DataFrame(instance_rows).to_csv(instance_path, index=False)
    pd.DataFrame(totals_rows).to_csv(totals_path, index=False)
    return instance_path, totals_path

In [8]:
baseline_instance, baseline_totals = run_reports(tag='baseline', conf=BEST_CONF, half=False)
fp16_instance, fp16_totals = run_reports(tag='fp16', conf=BEST_CONF, half=True)

print('Baseline instance:', baseline_instance)
print('Baseline totals:  ', baseline_totals)
print('FP16 instance:    ', fp16_instance)
print('FP16 totals:      ', fp16_totals)

Baseline instance: D:\projeto_placentas_clayton\dev\projeto-placentas\v2_yolo11s_opt_v1\artifacts\reports\placenta_instance_report_baseline.csv
Baseline totals:   D:\projeto_placentas_clayton\dev\projeto-placentas\v2_yolo11s_opt_v1\artifacts\reports\placenta_totals_report_baseline.csv
FP16 instance:     D:\projeto_placentas_clayton\dev\projeto-placentas\v2_yolo11s_opt_v1\artifacts\reports\placenta_instance_report_fp16.csv
FP16 totals:       D:\projeto_placentas_clayton\dev\projeto-placentas\v2_yolo11s_opt_v1\artifacts\reports\placenta_totals_report_fp16.csv


In [9]:
def save_iou_visualizations(tag: str, conf: float, half: bool) -> Path:
    model_local = YOLO(str(weights_path))
    results = model_local.predict(
        source=str(val_images),
        conf=conf,
        imgsz=CFG['img_w'],
        retina_masks=CFG['retina_masks'],
        half=half,
        verbose=False,
    )

    out_dir = out_iou / tag
    out_dir.mkdir(parents=True, exist_ok=True)

    for r in results:
        img_name = Path(r.path).name
        label_path = val_labels / (Path(img_name).stem + '.txt')

        img = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)
        gt_masks = parse_gt_masks(label_path, CFG['img_w'], CFG['img_h'])
        ai_masks = masks_from_result(r)

        gt_combined = np.zeros((CFG['img_h'], CFG['img_w']), dtype=np.uint8)
        ai_combined = np.zeros((CFG['img_h'], CFG['img_w']), dtype=np.uint8)
        for m in gt_masks:
            gt_combined = np.logical_or(gt_combined, m).astype(np.uint8)
        for m in ai_masks:
            ai_combined = np.logical_or(ai_combined, m).astype(np.uint8)

        inter = np.logical_and(gt_combined, ai_combined).sum()
        union = np.logical_or(gt_combined, ai_combined).sum()
        img_iou = (inter / union) if union > 0 else 0.0

        overlay = img.copy()
        overlay[gt_combined == 1] = (overlay[gt_combined == 1] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
        overlay[ai_combined == 1] = (overlay[ai_combined == 1] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
        overlap = np.logical_and(gt_combined, ai_combined)
        overlay[overlap] = (overlay[overlap] * 0.5 + np.array([255, 255, 0]) * 0.5).astype(np.uint8)

        diff = np.zeros((CFG['img_h'], CFG['img_w'], 3), dtype=np.uint8)
        diff[np.logical_and(gt_combined == 1, ai_combined == 0)] = [0, 255, 0]
        diff[np.logical_and(gt_combined == 0, ai_combined == 1)] = [255, 0, 0]
        diff[np.logical_and(gt_combined == 1, ai_combined == 1)] = [255, 255, 0]

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        fig.suptitle(f'{img_name} | IoU: {img_iou:.3f} | half={half}', fontsize=12)
        axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
        axes[1].imshow(overlay); axes[1].set_title('Overlay'); axes[1].axis('off')
        axes[2].imshow(diff); axes[2].set_title('Diff map'); axes[2].axis('off')

        save_path = out_dir / f'iou_viz_{Path(img_name).stem}.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)

    return out_dir


baseline_iou_dir = save_iou_visualizations('baseline', BEST_CONF, half=False)
fp16_iou_dir = save_iou_visualizations('fp16', BEST_CONF, half=True)
print('Baseline IoU dir:', baseline_iou_dir)
print('FP16 IoU dir:    ', fp16_iou_dir)

Baseline IoU dir: D:\projeto_placentas_clayton\dev\projeto-placentas\v2_yolo11s_opt_v1\artifacts\iou_viz\baseline
FP16 IoU dir:     D:\projeto_placentas_clayton\dev\projeto-placentas\v2_yolo11s_opt_v1\artifacts\iou_viz\fp16


In [10]:
def benchmark_inference(tag: str, conf: float, half: bool) -> Path:
    model_local = YOLO(str(weights_path))

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    t0 = time.time()
    results = model_local.predict(
        source=str(val_images),
        conf=conf,
        imgsz=CFG['img_w'],
        retina_masks=CFG['retina_masks'],
        half=half,
        verbose=False,
    )
    n_images = len(results)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        peak_mem_mb = None

    elapsed = time.time() - t0
    ips = (n_images / elapsed) if elapsed > 0 else 0.0

    payload = {
        'tag': tag,
        'half': half,
        'conf': conf,
        'n_images': n_images,
        'elapsed_sec': elapsed,
        'images_per_sec': ips,
        'peak_gpu_mem_mb': peak_mem_mb,
    }

    out_json = out_bench / f'benchmark_{tag}.json'
    out_json.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    return out_json


bench_base = benchmark_inference('baseline', BEST_CONF, half=False)
bench_fp16 = benchmark_inference('fp16', BEST_CONF, half=True)
print('Benchmark baseline:', bench_base)
print('Benchmark fp16:    ', bench_fp16)

Benchmark baseline: D:\projeto_placentas_clayton\dev\projeto-placentas\v2_yolo11s_opt_v1\artifacts\benchmarks\benchmark_baseline.json
Benchmark fp16:     D:\projeto_placentas_clayton\dev\projeto-placentas\v2_yolo11s_opt_v1\artifacts\benchmarks\benchmark_fp16.json


In [11]:
base = pd.read_csv(out_reports / 'placenta_totals_report_baseline.csv')
fp16 = pd.read_csv(out_reports / 'placenta_totals_report_fp16.csv')

merge = base[['Image', 'AI_Count', 'AI_Area_px']].merge(
    fp16[['Image', 'AI_Count', 'AI_Area_px']], on='Image', suffixes=('_base', '_fp16')
)
merge['count_delta'] = merge['AI_Count_fp16'] - merge['AI_Count_base']
merge['area_delta_px'] = merge['AI_Area_px_fp16'] - merge['AI_Area_px_base']

summary = {
    'images': int(len(merge)),
    'count_delta_abs_sum': int(merge['count_delta'].abs().sum()),
    'area_delta_abs_sum_px': int(merge['area_delta_px'].abs().sum()),
    'count_delta_mean': float(merge['count_delta'].mean()),
    'area_delta_mean_px': float(merge['area_delta_px'].mean()),
}

cmp_json = out_bench / 'baseline_vs_fp16_report_delta.json'
cmp_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(summary)
cmp_json

{'images': 27, 'count_delta_abs_sum': 131, 'area_delta_abs_sum_px': 188740, 'count_delta_mean': -4.851851851851852, 'area_delta_mean_px': -6968.222222222223}


WindowsPath('D:/projeto_placentas_clayton/dev/projeto-placentas/v2_yolo11s_opt_v1/artifacts/benchmarks/baseline_vs_fp16_report_delta.json')